In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 120.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 0. IMPORT LIBRARIES
# ============================================================
import os
import math
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch
import faiss

import logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [ ]:
# ============================================================
# 1. LOAD DATA DOKUMEN DARI GOOGLE SHEETS
# ============================================================

DATA_URL = "https://docs.google.com/spreadsheets/d/1gTmTtyRO5SmQsq4_QdB1ZnUdSUNmoNyL7MciKfVdgtE/export?format=csv&gid=527481706"

docs = pd.read_csv(DATA_URL)
print("Kolom dataset:", list(docs.columns))
print("Jumlah dokumen:", len(docs))

# doc_id = index (0..N-1), HARUS konsisten dengan qrels yang pakai index
docs = docs.reset_index(drop=True)
docs["doc_id"] = docs.index

# Kolom teks untuk semantic search
if "text_semantic" in docs.columns:
    TEXT_COL = "text_semantic"
elif "text_bm25" in docs.columns:
    TEXT_COL = "text_bm25"
else:
    raise ValueError("Tidak ada kolom 'text_semantic' atau 'text_bm25' di dataset.")

docs[TEXT_COL] = docs[TEXT_COL].fillna("")

print("\nPreview data:")
display(docs[["doc_id", "title", TEXT_COL]].head())


Kolom dataset: ['url', 'title', 'abstract', 'text_semantic', 'text_bm25', 'text_word2vec']
Jumlah dokumen: 15326

Preview data:


,doc_id,title,text_semantic
0,0,PENERAPAN METODE CLARKE AND WRIGHT SAVING HEUR...,penerapan metode clarke and wright saving heur...
1,1,OPINION SHOPPING SEBAGAI PEMODERASI PENGARUH F...,opinion shopping sebagai pemoderasi pengaruh f...
2,2,PREDIKSI PENYAKIT JANTUNG MENGGUNAKAN ALGORITM...,prediksi penyakit jantung menggunakan algoritm...
3,3,IDENTIFIKASI TANDA TANGAN DENGAN METODE CONVOL...,identifikasi tanda tangan dengan metode convol...
4,4,IDENTIFIKASI MATA UANG LOGAM MENGGUNAKAN SEGME...,identifikasi mata uang logam menggunakan segme...


In [ ]:
#detail MEMUAT MODEL INDOSBERT BASE

MODEL_NAME_BASE = "firqaaa/indo-sentence-bert-base"

print(f"Memuat base IndoSBERT: {MODEL_NAME_BASE}")
model_base = SentenceTransformer(MODEL_NAME_BASE)

print("Dimensi embedding:", model_base.get_sentence_embedding_dimension())


Memuat base IndoSBERT: firqaaa/indo-sentence-bert-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimensi embedding: 768


In [ ]:
# Pembentukan Embedding Dokumen (Baseline)

doc_texts = docs[TEXT_COL].fillna("").tolist()

embeddings_base = model_base.encode(
    doc_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Shape embedding baseline:", embeddings_base.shape)
print("Contoh embedding (5 dimensi):", embeddings_base[0][:5])


Batches:   0%|          | 0/240 [00:00<?, ?it/s]

Shape embedding baseline: (15326, 768)
Contoh embedding (5 dimensi): [-0.03269677  0.0644844   0.0121755  -0.05882969 -0.01111579]


In [ ]:
# Penyusunan Data Fine-Tuning (Title–Abstract)
# Cari kolom abstrak secara otomatis
cand_abs_cols = [c for c in docs.columns if ("abstrak" in c.lower()) or ("abstract" in c.lower())]
if not cand_abs_cols:
    raise ValueError("Tidak ditemukan kolom abstrak (nama kolom mengandung 'abstrak' atau 'abstract').")

ABSTRACT_COL = cand_abs_cols[0]
print("Kolom abstrak yang dipakai:", ABSTRACT_COL)

train_df = docs[["title", ABSTRACT_COL]].copy()
train_df["title"] = train_df["title"].astype(str).fillna("").str.strip()
train_df[ABSTRACT_COL] = train_df[ABSTRACT_COL].astype(str).fillna("").str.strip()

min_abs_len = 50
mask = (train_df["title"].str.len() > 5) & (train_df[ABSTRACT_COL].str.len() > min_abs_len)
train_df = train_df[mask].reset_index(drop=True)

print("Jumlah pasangan judul–abstrak:", len(train_df))

Kolom abstrak yang dipakai: abstract
Jumlah pasangan judul–abstrak: 14416


In [ ]:
# Finetuning INDOSBERT
# Initialize model_ft if not already defined
# This makes the cell executable in isolation, though the full fine-tuning logic is in cell vH66XP32_YGV.
if 'model_ft' not in locals() or model_ft is None:
    print(f"Initializing model_ft from base for this cell: {MODEL_NAME_BASE}")
    model_ft = SentenceTransformer(MODEL_NAME_BASE)

# train_examples is assumed to be defined by a previous cell (e.g., Ph8DT0zN_TDc)
# Removed redundant train_examples definition from here.

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model_ft)

model_ft.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 0.1),
    show_progress_bar=True
)

Jumlah train_examples: 14416
Contoh: <InputExample> label: 0, texts: PENERAPAN METODE CLARKE AND WRIGHT SAVING HEURISTIC DALAM MENENTUKAN RUTE PENDISTRIBUSIAN PRODUK DI BAGIAN DISTRIBUTOR KOPERASI KAREB BOJONEGORO; Koperasi Kareb Bojonegoro bagian distributor merupakan salah satu bagian dari unit kerja pertokoan. Koperasi Kareb Bojonegoro bagian distributor melayani pendistribusian produk seperti bahan makanan, minuman dan tissu. Permasalahan yang dihadapi oleh Koperasi Kareb adalah perusahaan sudah memiliki rute pendistribusian sendiri yang dilakukan berdasarkan pengalaman dan perkiraan salesman yang bekerja. Selain itu dalam rute satu perjalanan yang dimiliki perusahaan seharusnya masih bisa dimasukkan lagi pelanggan yang belum datang dengan melihat kapasitas kendaraan. Berdasarkan permasalahan tersebut maka dibuatlah penelitian ini menggunakan metode Clarke and Wright Saving Heuristic untuk melakukan penghematan penghematan jarak tempuh sehingga dapat menghasilkan rute distribusi te

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


Step,Training Loss
500,0.028500
1000,0.014200
1500,0.004500


In [ ]:
# Pembentukan Embedding Dokumen (Fine-Tuned)

embeddings_ft = model_ft.encode(
    doc_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Shape embedding fine-tuned:", embeddings_ft.shape)
print("Contoh embedding (5 dimensi):", embeddings_ft[0][:5])


In [ ]:
 # ============================================================
# 2. BASELINE: LOAD INDOSBERT BASE
# ============================================================
MODEL_NAME_BASE = "firqaaa/indo-sentence-bert-base"

print(f"Memuat base IndoSBERT: {MODEL_NAME_BASE}")
model_base = SentenceTransformer(MODEL_NAME_BASE)

# ============================================================
# 3. AUTO BUILD / LOAD FAISS INDEX (BASELINE)
# ============================================================
INDEX_DIR_BASE = "indexes_base"
EMB_PATH_BASE   = os.path.join(INDEX_DIR_BASE, "doc_embeddings.npy")
ID_PATH_BASE    = os.path.join(INDEX_DIR_BASE, "doc_ids.npy")
FAISS_PATH_BASE = os.path.join(INDEX_DIR_BASE, "semantic_index.faiss")

os.makedirs(INDEX_DIR_BASE, exist_ok=True)
FORCE_REBUILD_BASE = False  # set True kalau ganti TEXT_COL / ganti model

def load_or_build_faiss_base(model, docs, TEXT_COL):
    global index_base, embeddings_base, doc_ids_base

    if (not FORCE_REBUILD_BASE) and all(os.path.exists(p) for p in [EMB_PATH_BASE, ID_PATH_BASE, FAISS_PATH_BASE]):
        print("[BASE][LOAD] Memuat embeddings & FAISS index dari disk...")
        embeddings_base = np.load(EMB_PATH_BASE)
        doc_ids_base = np.load(ID_PATH_BASE).tolist()
        index_base = faiss.read_index(FAISS_PATH_BASE)
        print("[BASE][READY] FAISS index loaded:", index_base.ntotal, "dokumen")
        return index_base

    print("[BASE][BUILD] Membuat embeddings & FAISS index baru (BASE MODEL)...")
    doc_texts = docs[TEXT_COL].fillna("").tolist()
    doc_ids_base = docs["doc_id"].tolist()

    embeddings_base = model.encode(
        doc_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    dim = embeddings_base.shape[1]
    index_base = faiss.IndexFlatIP(dim)
    index_base.add(embeddings_base)

    print("[BASE] Ukuran index:", index_base.ntotal)

    np.save(EMB_PATH_BASE, embeddings_base)
    np.save(ID_PATH_BASE, np.array(doc_ids_base, dtype=np.int32))
    faiss.write_index(index_base, FAISS_PATH_BASE)
    print("[BASE][SAVE] Embeddings & FAISS index disimpan di folder:", INDEX_DIR_BASE)

    return index_base

index_base = load_or_build_faiss_base(model_base, docs, TEXT_COL)


Memuat base IndoSBERT: firqaaa/indo-sentence-bert-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[BASE][BUILD] Membuat embeddings & FAISS index baru (BASE MODEL)...


Batches:   0%|          | 0/240 [00:00<?, ?it/s]

[BASE] Ukuran index: 15326
[BASE][SAVE] Embeddings & FAISS index disimpan di folder: indexes_base


In [ ]:
# ============================================================
# 4. FUNGSI SEMANTIC SEARCH (BASELINE)
# ============================================================

def semantic_search_base(query: str, topk: int = 10):
    q_emb = model_base.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores, indices = index_base.search(q_emb, topk)
    idxs = indices[0]
    ranked_doc_ids = [doc_ids_base[i] for i in idxs]
    ranked_scores = scores[0].tolist()
    return ranked_doc_ids, ranked_scores

# ============================================================
# 5. CONTOH PENCARIAN DENGAN SEMANTIC SEARCH (BASELINE)
# ============================================================
pd.set_option("display.max_colwidth", None)

test_queries = [
    "sistem rekomendasi",
]

for q in test_queries:
    print("\n==============================")
    print("BASELINE - QUERY:", q)
    ids, scs = semantic_search_base(q, topk=10)
    df_res = docs[docs["doc_id"].isin(ids)][["doc_id", "title", "url"]].copy()
    rank_map = {doc_id: i+1 for i, doc_id in enumerate(ids)}
    df_res["rank"] = df_res["doc_id"].map(rank_map)
    df_res["score"] = df_res["doc_id"].map({i: s for i, s in zip(ids, scs)})
    df_res = df_res.sort_values("rank")
    print(df_res.to_string(index=False))



BASELINE - QUERY: sistem rekomendasi
 doc_id                                                                                                                                                                                         title                                      url  rank    score
  12979                                Pengujian Sistem Informasi Pelayanan Administrasi Terpadu (SIPANDU) Kecamatan Tulungagung menggunakan Black Box Testing dengan Metode Equivalence Partitioning  https://repository.upnjatim.ac.id/5284/     1 0.581659
   4708                                                                     Sistem Pendukung Keputusan Pemberian Bonus Karyawan CV NAWASENA BUMI PERSADA Menggunakan Kombinasi Metode SWARA dan SMART https://repository.upnjatim.ac.id/23309/     2 0.569329
   7561  SISTEM PENDUKUNG KEPUTUSAN PEMILIHAN SUPPLIER DALAM PENGADAAN BARANG MENGGUNAKAN METODE ANALYTICAL HIERARCHY PROCESS DAN SIMPLE ADDITIVE WEIGHTING (STUDI KASUS : APOTEK MERPATI 1 SIDOARJO)  h

In [ ]:
# ============================================================
# 6. SIAPKAN DATA TRAINING: PASANGAN JUDUL–ABSTRAK
# ============================================================

# Cari kolom abstrak secara otomatis
cand_abs_cols = [c for c in docs.columns if ("abstrak" in c.lower()) or ("abstract" in c.lower())]
if not cand_abs_cols:
    raise ValueError("Tidak ditemukan kolom abstrak (nama kolom mengandung 'abstrak' atau 'abstract').")

ABSTRACT_COL = cand_abs_cols[0]
print("Kolom abstrak yang dipakai:", ABSTRACT_COL)

train_df = docs[["title", ABSTRACT_COL]].copy()
train_df["title"] = train_df["title"].astype(str).fillna("").str.strip()
train_df[ABSTRACT_COL] = train_df[ABSTRACT_COL].astype(str).fillna("").str.strip()

# Buang yang kosong atau terlalu pendek
min_abs_len = 50
mask = (train_df["title"].str.len() > 5) & (train_df[ABSTRACT_COL].str.len() > min_abs_len)
train_df = train_df[mask].reset_index(drop=True)

print("Jumlah pasangan judul–abstrak setelah filter:", len(train_df))

train_examples = [
    InputExample(texts=[row["title"], row[ABSTRACT_COL]])
    for _, row in train_df.iterrows()
]

print("\nContoh 2 pasangan:")
for ex in train_examples[:2]:
    print("  - TITLE :", ex.texts[0][:80])
    print("    ABS   :", ex.texts[1][:80])
    print()


Kolom abstrak yang dipakai: abstract
Jumlah pasangan judul–abstrak setelah filter: 14416

Contoh 2 pasangan:
  - TITLE : PENERAPAN METODE CLARKE AND WRIGHT SAVING HEURISTIC DALAM MENENTUKAN RUTE PENDIS
    ABS   : Koperasi Kareb Bojonegoro bagian distributor merupakan salah satu bagian dari un

  - TITLE : OPINION SHOPPING SEBAGAI PEMODERASI PENGARUH FINANCIAL DISTRESS, AUDIT CLIENT TE
    ABS   : Penelitian ini bertujuan untuk membuktikan opinion shopping sebagai pengaruh mod



In [ ]:
# ============================================================
# 7. LOAD / FINE-TUNE INDOSBERT (TITLE–ABSTRACT)
# ============================================================

MODEL_NAME_BASE = "firqaaa/indo-sentence-bert-base"
FINETUNED_DIR = "indosbert_finetuned_title_abs"

if os.path.exists(FINETUNED_DIR):
    print(f"[FT][LOAD] Memuat model fine-tuned dari: {FINETUNED_DIR}")
    model_ft = SentenceTransformer(FINETUNED_DIR)
else:
    print(f"[FT][INIT] Memuat base IndoSBERT: {MODEL_NAME_BASE}")
    model_ft = SentenceTransformer(MODEL_NAME_BASE)

    batch_size = 16
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

    train_loss = losses.MultipleNegativesRankingLoss(model_ft)

    num_epochs = 2  # bisa dinaikkan 3 kalau GPU kuat
    warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

    print(f"[FT][TRAIN] Fine-tuning IndoSBERT selama {num_epochs} epoch "
          f"pada {len(train_examples)} pasangan...")
    model_ft.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=num_epochs,
        warmup_steps=warmup_steps,
        show_progress_bar=True
    )

    os.makedirs(FINETUNED_DIR, exist_ok=True)
    model_ft.save(FINETUNED_DIR)
    print(f"[FT][SAVE] Model fine-tuned disimpan di: {FINETUNED_DIR}")


[FT][INIT] Memuat base IndoSBERT: firqaaa/indo-sentence-bert-base


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2b325115-49fa-4e1d-8cc7-a9561e3ab5f5)')' thrown while requesting HEAD https://huggingface.co/firqaaa/indo-sentence-bert-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


[FT][TRAIN] Fine-tuning IndoSBERT selama 2 epoch pada 14416 pasangan...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
500,0.033800
1000,0.012700
1500,0.004700


[FT][SAVE] Model fine-tuned disimpan di: indosbert_finetuned_title_abs


In [ ]:
# ============================================================
# 8. AUTO BUILD / LOAD FAISS INDEX (FINE-TUNED)
# ============================================================

INDEX_DIR_FT = "indexes_finetuned"
EMB_PATH_FT   = os.path.join(INDEX_DIR_FT, "doc_embeddings.npy")
ID_PATH_FT    = os.path.join(INDEX_DIR_FT, "doc_ids.npy")
FAISS_PATH_FT = os.path.join(INDEX_DIR_FT, "semantic_index.faiss")

os.makedirs(INDEX_DIR_FT, exist_ok=True)
FORCE_REBUILD_FT = False  # set True kalau ganti TEXT_COL / ganti model_ft

def load_or_build_faiss_finetuned(model, docs, TEXT_COL):
    global index_ft, embeddings_ft, doc_ids_ft

    if (not FORCE_REBUILD_FT) and all(os.path.exists(p) for p in [EMB_PATH_FT, ID_PATH_FT, FAISS_PATH_FT]):
        print("[FT][LOAD] Memuat embeddings & FAISS index (fine-tuned) dari disk...")
        embeddings_ft = np.load(EMB_PATH_FT)
        doc_ids_ft = np.load(ID_PATH_FT).tolist()
        index_ft = faiss.read_index(FAISS_PATH_FT)
        print("[FT][READY] FAISS index loaded:", index_ft.ntotal, "dokumen")
        return index_ft

    print("[FT][BUILD] Membuat embeddings & FAISS index baru (FINE-TUNED MODEL)...")
    doc_texts = docs[TEXT_COL].fillna("").tolist()
    doc_ids_ft = docs["doc_id"].tolist()

    embeddings_ft = model.encode(
        doc_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        device=DEVICE,
    )

    dim = embeddings_ft.shape[1]
    index_ft = faiss.IndexFlatIP(dim)
    index_ft.add(embeddings_ft)

    print("[FT] Ukuran index:", index_ft.ntotal)

    np.save(EMB_PATH_FT, embeddings_ft)
    np.save(ID_PATH_FT, np.array(doc_ids_ft, dtype=np.int32))
    faiss.write_index(index_ft, FAISS_PATH_FT)
    print("[FT][SAVE] Embeddings & FAISS index disimpan di folder:", INDEX_DIR_FT)

    return index_ft

index_ft = load_or_build_faiss_finetuned(model_ft, docs, TEXT_COL)


[FT][BUILD] Membuat embeddings & FAISS index baru (FINE-TUNED MODEL)...


Batches:   0%|          | 0/240 [00:00<?, ?it/s]

[FT] Ukuran index: 15326
[FT][SAVE] Embeddings & FAISS index disimpan di folder: indexes_finetuned


In [ ]:
# ============================================================
# 9. FUNGSI SEMANTIC SEARCH (FINE-TUNED)
# ============================================================

def semantic_search_finetuned(query: str, topk: int = 10):
    q_emb = model_ft.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=DEVICE,
    )
    scores, indices = index_ft.search(q_emb, topk)
    idxs = indices[0]
    ranked_doc_ids = [doc_ids_ft[i] for i in idxs]
    ranked_scores = scores[0].tolist()
    return ranked_doc_ids, ranked_scores



In [ ]:
# ============================================================
# 10. CONTOH: BANDINGKAN BASELINE VS FINE-TUNED
# ============================================================

pd.set_option("display.max_colwidth", None)

test_queries = [
    "machine learning",
    "keamanan dan jaringan it",
    "sistem rekomendasi",
    "pemasaran digital",
]

for q in test_queries:
    print("\n==============================================")
    print("QUERY:", q)

    # BASELINE
    base_ids, base_scs = semantic_search_base(q, topk=5)
    base_df = docs[docs["doc_id"].isin(base_ids)][["doc_id", "title"]].copy()
    base_rank = {doc_id: i+1 for i, doc_id in enumerate(base_ids)}
    base_df["rank"] = base_df["doc_id"].map(base_rank)
    base_df = base_df.sort_values("rank")

    print("\n[BASELINE]")
    print(base_df.to_string(index=False))

    # FINE-TUNED
    ft_ids, ft_scs = semantic_search_finetuned(q, topk=5)
    ft_df = docs[docs["doc_id"].isin(ft_ids)][["doc_id", "title"]].copy()
    ft_rank = {doc_id: i+1 for i, doc_id in enumerate(ft_ids)}
    ft_df["rank"] = ft_df["doc_id"].map(ft_rank)
    ft_df = ft_df.sort_values("rank")

    print("\n[FINE-TUNED]")
    print(ft_df.to_string(index=False))



QUERY: machine learning

[BASELINE]
 doc_id                                                                                                                                                                        title  rank
  13757                                                                           E-Learning Berbasis Web untuk Meningkatkan Efektivitas Pembelajaran Secara Daring di Sekolah Dasar     1
    737    Pengembangan Aplikasi Pembelajaran Online Berbasis Web Dengan Metode Gamifikasi Menggunakan Framework Codeigniter Pada Program Studi Informatika UPN “Veteran” Jawa Timur     2
    529                                                                                                           Rancang Bangun Sistem Informasi Pembelajaran SMA Trimurti Surabaya     3
  14713                                                                               SISTEM APLIKASI ABSENSI MAHASISWA MENGGUNAKAN METODE QR CODE (QUICK RESPONSE) BERBASIS ANDROID     4
  14703 MEDIA EDUKASI “EXERX

In [ ]:
# 1. Tentukan nama folder yang ingin di-zip dan nama file zip keluaran
folder_to_zip = 'indexes_base' # Ganti dengan nama folder Anda
output_zip_name = 'indexes_base.zip'

# 2. Jalankan perintah zip menggunakan shell Linux
!zip -r {output_zip_name} {folder_to_zip}

# 3. Unduh file zip yang sudah dibuat ke komputer lokal Anda
from google.colab import files
files.download(output_zip_name)

  adding: indexes_base/ (stored 0%)
  adding: indexes_base/doc_embeddings.npy (deflated 7%)
  adding: indexes_base/doc_ids.npy (deflated 65%)
  adding: indexes_base/semantic_index.faiss (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Tentukan nama folder yang ingin di-zip dan nama file zip keluaran
folder_to_zip = 'indexes_finetuned' # Ganti dengan nama folder Anda
output_zip_name = 'indexes_finetuned.zip'

# 2. Jalankan perintah zip menggunakan shell Linux
!zip -r {output_zip_name} {folder_to_zip}

# 3. Unduh file zip yang sudah dibuat ke komputer lokal Anda
from google.colab import files
files.download(output_zip_name)

  adding: indexes_finetuned/ (stored 0%)
  adding: indexes_finetuned/doc_embeddings.npy (deflated 7%)
  adding: indexes_finetuned/doc_ids.npy (deflated 65%)
  adding: indexes_finetuned/semantic_index.faiss (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Tentukan nama folder yang ingin di-zip dan nama file zip keluaran
folder_to_zip = 'indosbert_finetuned_title_abs' # Ganti dengan nama folder Anda
output_zip_name = 'indosbert_finetuned_title_abs.zip'

# 2. Jalankan perintah zip menggunakan shell Linux
!zip -r {output_zip_name} {folder_to_zip}

# 3. Unduh file zip yang sudah dibuat ke komputer lokal Anda
from google.colab import files
files.download(output_zip_name)

  adding: indosbert_finetuned_title_abs/ (stored 0%)
  adding: indosbert_finetuned_title_abs/README.md (deflated 67%)
  adding: indosbert_finetuned_title_abs/tokenizer.json (deflated 71%)
  adding: indosbert_finetuned_title_abs/1_Pooling/ (stored 0%)
  adding: indosbert_finetuned_title_abs/1_Pooling/config.json (deflated 59%)
  adding: indosbert_finetuned_title_abs/vocab.txt (deflated 53%)
  adding: indosbert_finetuned_title_abs/modules.json (deflated 53%)
  adding: indosbert_finetuned_title_abs/special_tokens_map.json (deflated 80%)
  adding: indosbert_finetuned_title_abs/config.json (deflated 57%)
  adding: indosbert_finetuned_title_abs/model.safetensors (deflated 7%)
  adding: indosbert_finetuned_title_abs/config_sentence_transformers.json (deflated 40%)
  adding: indosbert_finetuned_title_abs/tokenizer_config.json (deflated 74%)
  adding: indosbert_finetuned_title_abs/sentence_bert_config.json (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>